# FRED - Final Hybrid Dataset Fine-Tuning
This notebook fine-tunes Qwen2.5-7B-Instruct using 4-bit QLoRA on the final 5,997-record dataset.


In [ ]:
%%capture
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps "xformers<0.0.27" "trl<0.9.0" peft accelerate bitsandbytes datasets


In [ ]:
import torch
from unsloth import FastLanguageModel
import json
import random

# Fixed random seed
random.seed(42)
torch.manual_seed(42)

max_seq_length = 2048
dtype = None # Auto detection
load_in_4bit = True

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Qwen2.5-7B-Instruct",
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
    token = None,
)


In [ ]:
# Configure LoRA adapters
model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj"],
    lora_alpha = 16,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 42,
    use_rslora = False,
    loftq_config = None,
)


## 1. Load and Verify Final Dataset


In [ ]:
from datasets import load_dataset, Dataset

# We assume this notebook runs in a Kaggle environment where the dataset is uploaded or mounted.
# Adjust paths if the dataset is located elsewhere in Kaggle (e.g., /kaggle/input/...)
TRAIN_PATH = "data/final/train_chatml.jsonl"
VAL_PATH = "data/final/validation_chatml.jsonl"

def load_jsonl(path):
    with open(path, "r", encoding="utf-8") as f:
        return [json.loads(line) for line in f if line.strip()]

train_data = load_jsonl(TRAIN_PATH)
val_data = load_jsonl(VAL_PATH)

print(f"Train rows: {len(train_data)}")
print(f"Validation rows: {len(val_data)}")

# Verify schema and data separation
assert all("messages" in r for r in train_data), "Train schema missing 'messages'"
assert all("messages" in r for r in val_data), "Val schema missing 'messages'"
assert len(train_data) == 4532, f"Expected 4532 train rows, got {len(train_data)}"
assert len(val_data) == 624, f"Expected 624 val rows, got {len(val_data)}"

# We ensure no test data is loaded by explicitly only loading train and val JSONLs.

# Display one clean and one corrupted sample from train
clean_sample = None
corrupted_sample = None
for r in train_data:
    target = r["messages"][1]["content"]
    if "<numerical>" in target or "<temporal>" in target:
        if corrupted_sample is None:
            corrupted_sample = r
    else:
        if clean_sample is None:
            clean_sample = r
    if clean_sample and corrupted_sample:
        break

print("\n--- Corrupted Sample ---")
print(json.dumps(corrupted_sample, indent=2))
print("\n--- Clean Sample ---")
print(json.dumps(clean_sample, indent=2))


## 2. Format ChatML for Unsloth


In [ ]:
from unsloth.chat_templates import get_chat_template

tokenizer = get_chat_template(
    tokenizer,
    chat_template = "chatml",
    mapping = {"role": "role", "content": "content", "user": "user", "assistant": "assistant"}
)

from unsloth.chat_templates import standardize_sharegpt
from unsloth.chat_templates import train_on_responses_only

def formatting_prompts_func(examples):
    texts = []
    for msgs in examples["messages"]:
        text = tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=False)
        texts.append(text)
    return {"text": texts}

train_dataset = Dataset.from_list(train_data)
val_dataset = Dataset.from_list(val_data)

train_dataset = train_dataset.map(formatting_prompts_func, batched=True)
val_dataset = val_dataset.map(formatting_prompts_func, batched=True)


In [ ]:
# Use response-only loss masking
train_dataset = train_on_responses_only(
    train_dataset,
    tokenizer = tokenizer,
    instruction_part = "<|im_start|>user\n",
    response_part = "<|im_start|>assistant\n",
)
val_dataset = train_on_responses_only(
    val_dataset,
    tokenizer = tokenizer,
    instruction_part = "<|im_start|>user\n",
    response_part = "<|im_start|>assistant\n",
)


## 3. Training Setup


In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments, DataCollatorForSeq2Seq
from unsloth import is_bfloat16_supported

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = train_dataset,
    eval_dataset = val_dataset,
    dataset_text_field = "text",
    max_seq_length = max_seq_length,
    data_collator = DataCollatorForSeq2Seq(tokenizer = tokenizer),
    dataset_num_proc = 2,
    packing = False, # Can make training 5x faster for short sequences.
    args = TrainingArguments(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps = 50,
        num_train_epochs = 3,
        learning_rate = 2e-5,
        fp16 = not is_bfloat16_supported(),
        bf16 = is_bfloat16_supported(),
        logging_steps = 10,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 42,
        output_dir = "outputs",
        evaluation_strategy = "epoch",
        save_strategy = "epoch",
        load_best_model_at_end = True,
        metric_for_best_model = "eval_loss",
        greater_is_better = False,
        save_total_limit = 2,
    ),
)


In [ ]:
# Run training
trainer_stats = trainer.train()


## 4. Smoke Tests


In [ ]:
# Switch model to inference mode
FastLanguageModel.for_inference(model)

import json
TEST_PATH = "data/final/test_chatml.jsonl"
test_data = load_jsonl(TEST_PATH)

# Collect 3 numerical, 3 temporal, 3 clean
test_samples = []
counts = {"numerical": 0, "temporal": 0, "clean": 0}

for r in test_data:
    target = r["messages"][1]["content"]
    if "<numerical>" in target and counts["numerical"] < 3:
        test_samples.append(("numerical", r))
        counts["numerical"] += 1
    elif "<temporal>" in target and counts["temporal"] < 3:
        test_samples.append(("temporal", r))
        counts["temporal"] += 1
    elif "<numerical>" not in target and "<temporal>" not in target and counts["clean"] < 3:
        test_samples.append(("clean", r))
        counts["clean"] += 1
        
    if all(v >= 3 for v in counts.values()):
        break

print("Running Smoke Tests...\n")

smoke_test_results = []

for category, sample in test_samples:
    messages = sample["messages"]
    prompt = tokenizer.apply_chat_template([messages[0]], tokenize=False, add_generation_prompt=True)
    inputs = tokenizer([prompt], return_tensors="pt").to("cuda")
    
    outputs = model.generate(**inputs, max_new_tokens=512, use_cache=True, temperature=0.0)
    response_text = tokenizer.batch_decode(outputs, skip_special_tokens=True)[0]
    
    # Extract only the generated assistant text
    generation = response_text.split("assistant\n")[-1].strip()
    
    expected = messages[1]["content"]
    
    print(f"[{category.upper()}]")
    print(f"EXPECTED:\n{expected}")
    print(f"GENERATED:\n{generation}\n")
    print("-" * 50)
    
    smoke_test_results.append({
        "category": category,
        "input": messages[0]["content"],
        "expected": expected,
        "generated": generation
    })



## 5. Save Artifacts


In [ ]:
# Save the LoRA adapter locally
model.save_pretrained("best_lora_adapter")
tokenizer.save_pretrained("best_lora_adapter")

# Save metrics and configs
with open("training_stats.json", "w") as f:
    json.dump(trainer_stats.metrics, f, indent=2)
    
with open("smoke_test_results.json", "w") as f:
    json.dump(smoke_test_results, f, indent=2)

print("Artifacts saved locally to best_lora_adapter/ and JSON files.")


## 6. Hugging Face Upload (Explicit)
Only run this cell if training succeeded and you wish to push to HF.


In [ ]:
import os
from kaggle_secrets import UserSecretsClient

try:
    user_secrets = UserSecretsClient()
    hf_token = user_secrets.get_secret("HF_TOKEN")
except Exception as e:
    print("Could not retrieve HF_TOKEN from Kaggle secrets. Make sure it is defined.")
    hf_token = None

if hf_token:
    # Explicitly define repository ID
    hf_repo_id = input("Enter your Hugging Face Repository ID (e.g., your_username/fred-qwen2.5-7b-lora): ")
    
    if hf_repo_id:
        print(f"Pushing to Hub: {hf_repo_id}...")
        
        # We push only the LoRA adapters, not the full model, keeping it lightweight
        model.push_to_hub(hf_repo_id, token=hf_token)
        tokenizer.push_to_hub(hf_repo_id, token=hf_token)
        
        # Create and upload the Model Card
        model_card = f"""---
tags:
- finqa
- hallucination-detection
- finance
- unsloth
- qlora
- qwen2.5
---

# FRED - Resource-Constrained Reproduction

This is a 4-bit QLoRA adapter for **Qwen2.5-7B-Instruct** trained on the FRED Hybrid dataset to perform financial hallucination detection and correction.

## Training Details
- **Total Records:** 5,997 (4,532 Train / 624 Val / 841 Test)
- **Composition:** 4,747 Corrupted examples (447 LLM Gemini-generated seed, 4,300 Deterministic Python-generated) and 1,250 Clean pass-through examples.
- **Taxonomy Focus:** Numerical and Temporal errors.
- **Base Model:** Qwen/Qwen2.5-7B-Instruct
- **Fine-tuning Strategy:** 4-bit QLoRA (Rank 16, Alpha 16)
- **Epochs:** 3
- **Learning Rate:** 2e-5
- **Max Sequence Length:** 2048
- **Data format:** ChatML with response-only loss masking.

## Formatting Requirement
The model was trained **without** free-form Chain-of-Thought at inference. It responds directly with the correction block using the format:
`Correction: [full response with exactly one <numerical> or <temporal> edit]`

## Limitations
- Only trained on financial QA domain data (FinQA / TAT-QA structure).
- Scoped exclusively to Numerical and Temporal error correction.
- The base model is 7B parameters and quantized to 4-bit, which may impact general reasoning outside its fine-tuned scope.
"""
        with open("README.md", "w") as f:
            f.write(model_card)
            
        from huggingface_hub import HfApi
        api = HfApi(token=hf_token)
        api.upload_file(
            path_or_fileobj="README.md",
            path_in_repo="README.md",
            repo_id=hf_repo_id,
        )
        print("Model and README pushed successfully!")
    else:
        print("Upload cancelled (no repo id provided).")
else:
    print("Cannot push without a valid HF_TOKEN.")

